# Chapter 11 — Efficient Streaming DataLoader

Data is the fuel of large language models, and its quality matters far more than its quantity. A model trained on 1 trillion tokens of noisy web scrapes will underperform one trained on 500 billion carefully curated tokens. This chapter covers the entire data pipeline — from collection to cleaning to loading — that determines your model's ceiling.

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import numpy as np
import tempfile
import os

class TokenDataset(IterableDataset):
    def __init__(self, data_path, seq_length):
        self.data = np.memmap(data_path, dtype=np.uint16, mode='r')
        self.seq_length = seq_length

    def __iter__(self):
        while True:
            idx = torch.randint(len(self.data) - self.seq_length - 1, (1,)).item()
            chunk = self.data[idx:idx + self.seq_length + 1]
            x = torch.from_numpy(chunk[:-1].astype(np.int64))
            y = torch.from_numpy(chunk[1:].astype(np.int64))
            yield x, y

vocab_size, num_tokens, seq_length = 32000, 10000, 64
tokens = np.random.randint(0, vocab_size, size=num_tokens, dtype=np.uint16)
tmp_path = tempfile.mktemp(suffix='.bin')
tokens.tofile(tmp_path)
print(f"Created {num_tokens:,} synthetic tokens ({os.path.getsize(tmp_path) / 1024:.1f} KB)")

loader = DataLoader(TokenDataset(tmp_path, seq_length), batch_size=4)
for i, (x, y) in enumerate(loader):
    print(f"\nBatch {i}:")
    print(f"  x shape: {tuple(x.shape)}, y shape: {tuple(y.shape)}")
    print(f"  x[0,:5]: {x[0,:5].tolist()}")
    print(f"  y[0,:5]: {y[0,:5].tolist()}")
    print(f"  x[0,1:4] == y[0,0:3]? {(x[0,1:4] == y[0,0:3]).all().item()} (shifted by 1)")
    if i >= 1:
        break

os.unlink(tmp_path)

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import numpy as np

class TokenDataset(IterableDataset):
    def __init__(self, data_path, seq_length):
        self.data = np.memmap(data_path, dtype=np.uint16, mode='r')
        self.seq_length = seq_length

    def __iter__(self):
        while True:
            idx = torch.randint(len(self.data) - self.seq_length - 1, (1,)).item()
            chunk = self.data[idx:idx + self.seq_length + 1]
            x = torch.from_numpy(chunk[:-1].astype(np.int64))
            y = torch.from_numpy(chunk[1:].astype(np.int64))
            yield x, y

loader = DataLoader(TokenDataset('train.bin', 1024), batch_size=32, num_workers=4, pin_memory=True)

---

**Course:** Aprenda LLM | **Chapter 11**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.